# 3.0 Feature engineering and SageMaker Feature Store (run in SageMaker Studio)

Builds 512-sample / 50%-overlap window features from curated Parquet and ingests them into an **offline-only** feature group (record id `window_id`). Uses existing `LabRole`; no IAM changes.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import boto3
from src.pamap2.config import load_config, s3_uri
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
import re, io, time
import pandas as pd
from src.pamap2 import features as F
s3 = session.client("s3")
FG_NAME = cfg["feature_group"]
print("Feature group:", FG_NAME)

## Build window features per curated partition

In [ ]:
prefix = f"{cfg['s3_prefixes']['curated']}pamap2/"
keys = []
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=cfg["bucket"], Prefix=prefix):
    keys += [o["Key"] for o in page.get("Contents", []) if o["Key"].endswith(".parquet")]
pat = re.compile(r"session=([^/]+)/subject_id=(\d+)/")
parts = []
for k in keys:
    m = pat.search(k)
    if m is None:
        print("Skipping non-matching key:", k)
        continue
    parts.append((m.group(1), int(m.group(2)), k))
parts.sort()
INGEST_TIME = time.time()   # one event_time per run
print(len(parts), "curated partitions")
frames = []
for sess, subj, key in parts:
    raw = pd.read_parquet(io.BytesIO(s3.get_object(Bucket=cfg["bucket"], Key=key)["Body"].read()))
    fr = F.build_feature_frame(raw, subj, sess, ingest_time=INGEST_TIME)
    print(f"{sess}/{subj}: {len(raw)} rows -> {len(fr)} windows")
    frames.append(fr)
feat = pd.concat(frames, ignore_index=True)
assert feat["window_id"].is_unique
print("Total windows:", len(feat), "| features:", len(F.FEATURE_COLUMNS))

In [ ]:
from src.pamap2.schema import ACTIVITY_NAMES
display(feat.groupby("activity_id").size().rename("windows").to_frame()
        .assign(activity=lambda d: d.index.map(ACTIVITY_NAMES)))
display(feat.pivot_table(index="subject_id", columns="activity_id", values="window_id", aggfunc="count").fillna(0).astype(int))

## Create (or reuse) the feature group - offline store only

In [ ]:
import sagemaker
from sagemaker.feature_store.feature_group import FeatureGroup
from botocore.exceptions import ClientError
sm = session.client("sagemaker")
fs_rt = session.client("sagemaker-featurestore-runtime")
sm_session = sagemaker.session.Session(boto_session=session, sagemaker_client=sm,
                                       sagemaker_featurestore_runtime_client=fs_rt)
fg = FeatureGroup(name=FG_NAME, sagemaker_session=sm_session)
fg.load_feature_definitions(data_frame=feat)
wanted = {d.feature_name: str(d.feature_type.value if hasattr(d.feature_type, "value") else d.feature_type)
          for d in fg.feature_definitions}

def describe():
    try:
        return sm.describe_feature_group(FeatureGroupName=FG_NAME)
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFound":
            return None
        raise

desc = describe()
if desc is None:
    fg.create(s3_uri=s3_uri(cfg, "features", "offline-store/"), record_identifier_name="window_id",
              event_time_feature_name="event_time", role_arn=cfg["role_arn"],
              enable_online_store=False)
    print("Creating feature group...")
else:
    existing = {f["FeatureName"]: f["FeatureType"] for f in desc["FeatureDefinitions"]}
    if existing != wanted:
        raise RuntimeError(f"Existing feature group {FG_NAME} has a different schema; "
                           f"missing={set(wanted)-set(existing)} extra={set(existing)-set(wanted)}")
    print("Reusing existing feature group (schema matches).")
deadline = time.time() + 30 * 60
while True:
    status = sm.describe_feature_group(FeatureGroupName=FG_NAME)["FeatureGroupStatus"]
    if status == "Created":
        break
    if status in ("Deleting", "CreateFailed", "DeleteFailed"):
        raise RuntimeError(f"Feature group status: {status}")
    if time.time() > deadline:
        raise TimeoutError(f"Feature group not Created after 30 min (status {status})")
    time.sleep(5)
print("Feature group status:", status)

## Ingest (upsert by `window_id`)

In [ ]:
res = fg.ingest(data_frame=feat, max_workers=4, wait=True)
assert not res.failed_rows, f"Ingest failed rows: {res.failed_rows}"
print("Ingested", len(feat), "records")

## Verify record count on the offline table

The offline store lags ingestion by several minutes; this polls for up to 20 minutes.

In [ ]:
import io, time, pandas as pd
athena = session.client("athena")
s3 = session.client("s3")
def run(sql, timeout=600, database=None):
    qid = athena.start_query_execution(QueryString=sql, WorkGroup=cfg["athena_workgroup"],
        QueryExecutionContext={"Database": database or cfg["glue_database"]})["QueryExecutionId"]
    deadline = time.time() + timeout
    while True:
        qe = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        st = qe["Status"]
        if st["State"] in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        if time.time() > deadline:
            athena.stop_query_execution(QueryExecutionId=qid)
            raise TimeoutError(f"Athena query exceeded {timeout}s: {sql[:200]}")
        time.sleep(1)
    if st["State"] != "SUCCEEDED":
        raise RuntimeError(f"{st['State']}: {st.get('StateChangeReason')}\n{sql}")
    if qe.get("StatementType") != "DML":
        return None
    out = qe["ResultConfiguration"]["OutputLocation"]
    b, k = out[len("s3://"):].split("/", 1)
    return pd.read_csv(io.BytesIO(s3.get_object(Bucket=b, Key=k)["Body"].read()))
d = sm.describe_feature_group(FeatureGroupName=FG_NAME)
dc = d["OfflineStoreConfig"]["DataCatalogConfig"]
tbl = f'"{dc["Database"]}"."{dc["TableName"]}"'
print("Offline table:", tbl)
ids = set(feat["window_id"])
n = total = 0
for _ in range(40):
    stored = run(f"SELECT DISTINCT window_id FROM {tbl}", database=dc["Database"]).window_id
    total = len(stored)
    n = int(stored.isin(ids).sum())
    if n >= len(feat):
        break
    time.sleep(30)
print("Current-run windows in offline store:", n, "| generated:", len(feat),
      "| extra (stale) window_ids:", total - n)
assert n == len(feat), "offline store is missing windows from this run (wait and re-run this cell)"